# KG-guided RAG for diagnosis on DDXPlus — baseline, ablations and our improvements

This notebook re-implements a simplified version of the pipeline from *MedRAG: Enhancing RAG with Knowledge Graph-Elicited Reasoning* (Zhao et al., arXiv:2502.04413) on the public **DDXPlus** dataset, and tests changes of our own. Everything runs locally and free: Ollama, sentence-transformers, bm25s, scikit-learn.

**Methods compared (same patients, same LLM)**

| Method | Retrieval of similar patients | Knowledge graph (KG) facts | What it tests |
|---|---|---|---|
| `closed_book` | none | none | LLM alone |
| `naive_rag` | dense | none | plain RAG |
| `kg_rag_paper_style` | dense | LLM-written features, unchecked | the paper's idea (simplified) |
| `hybrid_rag` | dense + BM25 (RRF) | none | **our change 1**: better case retrieval |
| `kg_rag_verified` | dense | features checked against textbooks | **our change 2**: verified KG |
| `ours_full` | dense + BM25 (RRF) | features checked against textbooks | both changes |

Each change is tested on its own and combined, so any gain can be attributed.

**Read this before comparing with the paper**
- Only **L3 accuracy** (the exact diagnosis, one of the DDXPlus pathologies) is comparable with the paper. Their L1/L2 groups are not published, so our L1/L2 use our own clusters.
- The paper's LLM and prompts differ from ours, and their test set is 30 patients per pathology. With a small test set the confidence intervals are wide.
- This is a simplified re-implementation, not their code. Treat differences as indicative, not as a proof of improvement.

**Data (manual download, about 1 GB):** get the **English** DDXPlus files from the link in the README at https://github.com/mila-iqia/ddxplus. Put `release_evidences.json`, `release_train_patients.*` and `release_test_patients.*` (zip or csv) in a folder named `ddxplus` next to this notebook. DDXPlus is synthetic, CC-BY, for research use only.

**Optional:** `medrag_chunks.jsonl` (your textbook corpus) enables the verified-KG methods. Without it those two methods fall back to unverified features and the notebook says so.

## 1. Install

In [ ]:
%pip install -q -U sentence-transformers bm25s PyStemmer ollama pandas numpy scikit-learn scipy matplotlib tqdm

## 2. Settings
The defaults give a **small but complete first run**. For final numbers set `TEST_PER_PATHOLOGY = 30` (the paper's protocol); it takes many hours on a CPU.
If you change any KG or prompt setting later, delete the files `cache_ddx/preds_*` so old predictions are not reused.

In [ ]:
import os, re, json, time, random, glob, ast
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from tqdm.auto import tqdm

# ---- files ----
DDX_DIR     = "ddxplus"
CHUNKS_PATH = "medrag_chunks.jsonl"     # optional, used to verify KG features
CACHE_DIR   = "cache_ddx"
RESULTS_DIR = "results_ddx"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---- models ----
LLM_MODEL   = "qwen3:4b"                # use a name shown by `ollama list`
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
THINK       = False                     # keep False on CPU
REASON_FIRST = False                    # True = the LLM writes a short reasoning before answering (slower, may help)

# ---- data sizes ----
POOL_PER_PATHOLOGY = 240    # retrievable patients per pathology (paper: 240)
TEST_PER_PATHOLOGY = 3      # test patients per pathology (paper: 30). 1 = quick smoke test
N_TEST_LIMIT       = None   # optional hard cap on test patients, e.g. 100
SEED               = 42

# ---- RAG / KG settings ----
TOP_K_CASES      = 5        # similar patients shown to the LLM
N_L2, N_L1       = 12, 5    # our disease clusters (sub-categories and categories)
LINK_MIN_FREQ    = 0.2      # a feature is linked to a disease if >=20% of its patients have it
N_AUG            = 8        # LLM-written key features per disease
MATCH_THRESHOLD  = 0.6      # similarity needed to match a free-text feature to a KG node
KG_TOP_CLUSTERS  = 1        # how many sub-categories feed the KG facts (paper: 1)
USE_TEXTBOOK_VERIFICATION = True

METHOD_CONFIG = {
    "closed_book":        {"cases": None,     "kg": None},
    "naive_rag":          {"cases": "dense",  "kg": None},
    "kg_rag_paper_style": {"cases": "dense",  "kg": "unverified"},
    "hybrid_rag":         {"cases": "hybrid", "kg": None},
    "kg_rag_verified":    {"cases": "dense",  "kg": "verified"},
    "ours_full":          {"cases": "hybrid", "kg": "verified"},
}
METHODS = list(METHOD_CONFIG)       # remove names here to run fewer methods

random.seed(SEED); np.random.seed(SEED)

## 3. Load DDXPlus and turn patients into text
Each patient has a list of evidence codes (symptoms and antecedents). We translate each code into a short English sentence such as *"Do you have a fever? Yes"*.

In [ ]:
def find_file(*patterns):
    for pattern in patterns:
        hits = [h for h in glob.glob(os.path.join(DDX_DIR, "**", pattern), recursive=True) if os.path.isfile(h)]
        if hits:
            return hits[0]
    raise FileNotFoundError(f"Could not find {patterns} inside '{DDX_DIR}'. See the data instructions at the top of the notebook.")

evidences  = json.load(open(find_file("release_evidences.json"), encoding="utf-8"))
train_path = find_file("release_train_patients*", "train.csv")
test_path  = find_file("release_test_patients*", "test.csv")

def clean_question(q):
    return re.sub(r"\s*\(.*?\)", "", q).strip()          # drop parentheses to keep prompts short

def patient_items(evidence_list):
    # one short sentence per evidence code, e.g. "E_55_@_V_167" or "E_91"
    items = []
    for code_str in evidence_list:
        name, _, value = code_str.partition("_@_")
        info = evidences.get(name)
        if info is None:
            continue
        question = clean_question(info.get("question_en") or info.get("question_fr") or name)
        if value:
            meaning = info.get("value_meaning", {}).get(value, {}).get("en", value)
            items.append(f"{question} {meaning}")
        else:
            items.append(f"{question} Yes")
    return items

def patient_text(row, items):
    sex = {"M": "Male", "F": "Female"}.get(row["SEX"], row["SEX"])
    return f"{sex}, {row['AGE']} years old. " + "; ".join(items)

def take_per_group(df, n, seed):
    parts = [g.sample(min(len(g), n), random_state=seed) for _, g in df.groupby("PATHOLOGY")]
    return pd.concat(parts)

# retrieval pool: scan the big train file in chunks and keep a random sample per pathology
parts = []
for chunk in tqdm(pd.read_csv(train_path, chunksize=200_000, usecols=["AGE", "SEX", "PATHOLOGY", "EVIDENCES"]),
                  desc="Scanning train file"):
    for _, g in chunk.groupby("PATHOLOGY"):
        parts.append(g.sample(min(len(g), POOL_PER_PATHOLOGY), random_state=SEED))
pool_df = take_per_group(pd.concat(parts), POOL_PER_PATHOLOGY, SEED).reset_index(drop=True)

# test set: balanced per pathology, shuffled so any prefix is a random sample
test_all = pd.read_csv(test_path, usecols=["AGE", "SEX", "PATHOLOGY", "EVIDENCES"])
test_all["orig_idx"] = test_all.index
test_df = take_per_group(test_all, TEST_PER_PATHOLOGY, SEED).sample(frac=1, random_state=SEED).reset_index(drop=True)
if N_TEST_LIMIT:
    test_df = test_df.iloc[:N_TEST_LIMIT].reset_index(drop=True)

for df in (pool_df, test_df):
    df["items"] = [patient_items(ast.literal_eval(e)) for e in df["EVIDENCES"]]
    df["text"]  = [patient_text(r, it) for (_, r), it in zip(df.iterrows(), df["items"])]

diseases  = sorted(pool_df["PATHOLOGY"].unique())
dis_index = {d: i for i, d in enumerate(diseases)}
print(f"{len(diseases)} pathologies | retrieval pool: {len(pool_df):,} patients | test set: {len(test_df)} patients")
print("\nExample patient (true diagnosis:", test_df.loc[0, "PATHOLOGY"], ")\n", test_df.loc[0, "text"][:600])

## 4. Embeddings and keyword index for similar-patient search
`dense` = meaning-based search with embeddings. `hybrid` = dense + BM25 keyword search merged with Reciprocal Rank Fusion (RRF).

In [ ]:
from sentence_transformers import SentenceTransformer
import bm25s, Stemmer

embedder = SentenceTransformer(EMBED_MODEL)
embedder.max_seq_length = 256

def embed(texts, desc="Embedding", block=1000):
    out = []
    for s in tqdm(range(0, len(texts), block), desc=desc, disable=len(texts) <= block):
        out.append(embedder.encode(texts[s:s + block], batch_size=64, normalize_embeddings=True, show_progress_bar=False))
    return np.vstack(out).astype("float32")

pool_emb_file = f"{CACHE_DIR}/pool_emb_{POOL_PER_PATHOLOGY}_{SEED}_{len(pool_df)}.npy"
if os.path.exists(pool_emb_file):
    pool_emb = np.load(pool_emb_file)
else:
    pool_emb = embed(pool_df["text"].tolist(), desc="Embedding pool patients")
    np.save(pool_emb_file, pool_emb)
test_emb = embed(test_df["text"].tolist(), desc="Embedding test patients")

stemmer = Stemmer.Stemmer("english")
bm25 = bm25s.BM25()
bm25.index(bm25s.tokenize(pool_df["text"].tolist(), stopwords="en", stemmer=stemmer, show_progress=False), show_progress=False)

def dense_search(pos, k):
    scores = pool_emb @ test_emb[pos]
    top = np.argpartition(-scores, k)[:k]
    return top[np.argsort(-scores[top])].tolist()

def bm25_search(pos, k):
    q = bm25s.tokenize(test_df.loc[pos, "text"], stopwords="en", stemmer=stemmer, show_progress=False)
    docs, _ = bm25.retrieve(q, k=k, show_progress=False)
    return docs[0].tolist()

def rrf(rank_lists, k=60):
    scores = defaultdict(float)
    for ranking in rank_lists:
        for rank, idx in enumerate(ranking):
            scores[idx] += 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

def retrieve_cases(pos, mode, k=TOP_K_CASES):
    if mode == "dense":
        return dense_search(pos, k)
    if mode == "bm25":
        return bm25_search(pos, k)
    return rrf([dense_search(pos, 50), bm25_search(pos, 50)])[:k]       # hybrid

### Quick check without the LLM: do similar patients share the diagnosis?
A k-nearest-neighbour vote (most common diagnosis among the retrieved patients) shows how good each retriever is on its own. This is fast and needs no LLM.

In [ ]:
def knn_vote(pos, mode, k=TOP_K_CASES):
    votes = Counter()
    for rank, idx in enumerate(retrieve_cases(pos, mode, k)):
        votes[pool_df.loc[idx, "PATHOLOGY"]] += 1 + 0.01 * (k - rank)      # tiny tie-break by rank
    return votes.most_common(1)[0][0]

knn_rows = []
for mode in ["dense", "bm25", "hybrid"]:
    preds = [knn_vote(p, mode) for p in tqdm(range(len(test_df)), desc=f"kNN vote ({mode})")]
    knn_rows.append({"retriever": mode, "kNN top-1 accuracy": np.mean(np.array(preds) == test_df["PATHOLOGY"].values)})
knn_df = pd.DataFrame(knn_rows).round(3)
knn_df

## 5. Build the diagnostic knowledge graph
Following the paper's four tiers:
- **L3**: the diseases. **L2 / L1**: groups of diseases with similar presentations (we cluster diseases by how often each feature occurs).
- **L4d**: features found in the patient records (the *nodes* we match new patients against).
- **L4a**: key features written by the LLM for each disease (the facts we give the LLM at diagnosis time).

In [ ]:
from sklearn.cluster import AgglomerativeClustering

node_names = sorted({f for items in pool_df["items"] for f in items})          # L4d nodes
node_index = {f: i for i, f in enumerate(node_names)}
node_emb   = embed(node_names, desc="Embedding feature nodes")

freq = np.zeros((len(diseases), len(node_names)), dtype="float32")             # P(feature | disease)
for items, d in zip(pool_df["items"], pool_df["PATHOLOGY"]):
    for f in set(items):
        freq[dis_index[d], node_index[f]] += 1
freq /= pool_df["PATHOLOGY"].value_counts().reindex(diseases).values[:, None]

n_l2 = min(N_L2, len(diseases) - 1)
n_l1 = min(N_L1, n_l2 - 1) if n_l2 > 2 else 1
l2_of = AgglomerativeClustering(n_clusters=n_l2, metric="cosine", linkage="average").fit_predict(freq)
centroids = np.vstack([freq[l2_of == c].mean(axis=0) for c in range(n_l2)])
l1_of_cluster = AgglomerativeClustering(n_clusters=n_l1, metric="cosine", linkage="average").fit_predict(centroids)
l1_of = l1_of_cluster[l2_of]

print(f"{len(node_names)} feature nodes | {n_l2} sub-categories (L2) | {n_l1} categories (L1)\n")
for c in range(n_l2):
    print(f"L2 group {c}:", ", ".join(d for d, k in zip(diseases, l2_of) if k == c))

### 5a. Connect to Ollama (needed from here on)

In [ ]:
import ollama

try:
    ollama.show(LLM_MODEL)
    print(f"Ollama OK, model '{LLM_MODEL}' is available")
except Exception as e:
    raise RuntimeError(f"Ollama problem: {e}\nStart the server (`ollama serve`) and make sure `ollama list` shows {LLM_MODEL}.")

def llm_json(prompt, schema=None, max_tokens=150):
    # asks Ollama for JSON; with a schema the output is forced to be valid, which also stops the model from rambling
    def call(fmt):
        reply = ollama.chat(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}], think=THINK, format=fmt,
                            options={"temperature": 0, "num_predict": max_tokens + (3000 if THINK else 0), "num_ctx": 6144})
        text = reply["message"]["content"]
        if "</think>" in text:
            text = text.split("</think>")[-1]
        return json.loads(text.strip())
    try:
        return call(schema if schema else "json")
    except Exception:
        try:
            return call("json")              # fallback for older servers without schema support
        except Exception:
            return None

print(llm_json("Reply as JSON: {\"ok\": true}"))

### 5b. L4a: LLM-written key features per disease (cached)

In [ ]:
aug_file = f"{CACHE_DIR}/augmented_features.json"
if os.path.exists(aug_file):
    augmented = json.load(open(aug_file))
else:
    augmented = {}
    for d in tqdm(diseases, desc="Writing key features"):
        prompt = (f'You are a medical expert. List {N_AUG} short clinical features (symptoms, signs, risk factors or typical findings) '
                  f'that best distinguish "{d}" from other diseases with similar presentations. '
                  f'Max 12 words per feature. Reply as JSON: {{"features": ["...", "..."]}}')
        data = llm_json(prompt, max_tokens=300)
        feats = data.get("features", []) if isinstance(data, dict) else []
        augmented[d] = [str(f) for f in feats][:N_AUG]
    json.dump(augmented, open(aug_file, "w"), indent=1)
print("Example:", diseases[0], "->", augmented[diseases[0]][:3])

### 5c. Our change: verify each LLM-written feature against the textbooks
The LLM can write wrong or invented features. For each one we retrieve textbook passages (BM25 over your corpus) and ask the LLM whether the passages support the claim. Unsupported features are removed from the *verified* KG.
This step is slow (hundreds of LLM calls) and cached.

In [ ]:
ver_file = f"{CACHE_DIR}/verified_features.json"
SUPPORT_SCHEMA = {"type": "object", "properties": {"supported": {"type": "boolean"}}, "required": ["supported"]}

if os.path.exists(ver_file):
    verified = json.load(open(ver_file))
elif USE_TEXTBOOK_VERIFICATION and os.path.exists(CHUNKS_PATH):
    tb_texts = [json.loads(line)["text"] for line in open(CHUNKS_PATH, encoding="utf-8")]
    tb_bm25 = bm25s.BM25()
    tb_bm25.index(bm25s.tokenize(tb_texts, stopwords="en", stemmer=stemmer, show_progress=True), show_progress=True)

    def textbook_search(query, k=3):
        q = bm25s.tokenize(query, stopwords="en", stemmer=stemmer, show_progress=False)
        docs, _ = tb_bm25.retrieve(q, k=k, show_progress=False)
        return [tb_texts[i] for i in docs[0]]

    verified = {}
    for d in tqdm(diseases, desc="Verifying features against textbooks"):
        keep = []
        for feat in augmented[d]:
            context = "\n\n".join(t[:600] for t in textbook_search(f"{d} {feat}"))
            prompt = (f'Textbook excerpts:\n{context}\n\nClaim: "{feat}" is a typical clinical feature of {d}.\n'
                      f'Do the excerpts support this claim? Reply as JSON: {{"supported": true}} or {{"supported": false}}')
            ans = llm_json(prompt, schema=SUPPORT_SCHEMA, max_tokens=15)
            if isinstance(ans, dict) and ans.get("supported") is True:
                keep.append(feat)
        verified[d] = keep
    json.dump(verified, open(ver_file, "w"), indent=1)
else:
    print("Textbook verification skipped (disabled or corpus file missing): 'verified' = unverified features.")
    verified = augmented

n_aug, n_ver = sum(len(v) for v in augmented.values()), sum(len(v) for v in verified.values())
print(f"Features kept after verification: {n_ver} of {n_aug} ({n_ver / max(n_aug, 1):.0%})")

## 6. Diagnostic-differences search (paper section 4.2, simplified)
1. Match the patient's features to KG nodes (exact match; embedding match with a threshold for unseen text).
2. Each matched node votes for the L2 sub-categories of the diseases it is linked to (a vote is split if it reaches several).
3. The winning sub-category's diseases and their LLM-written features become the "diagnostic differences" given to the LLM.

`routing accuracy` below shows how often the true disease sits in the chosen sub-category. This is a free check of the KG module.

In [ ]:
def match_nodes(feats):
    ids     = [node_index[f] for f in feats if f in node_index]          # exact matches
    missing = [f for f in feats if f not in node_index]                  # unseen text: match by embedding
    if missing:
        sims = embed(missing) @ node_emb.T
        ids += [int(b) for b, s in zip(sims.argmax(1), sims.max(1)) if s > MATCH_THRESHOLD]
    return ids

def route_clusters(feats, top=KG_TOP_CLUSTERS):
    votes = np.zeros(n_l2)
    for j in match_nodes(feats):
        linked = np.where(freq[:, j] >= LINK_MIN_FREQ)[0]
        if len(linked) == 0:
            continue
        clusters = set(l2_of[linked].tolist())
        for c in clusters:
            votes[c] += 1.0 / len(clusters)
    return np.argsort(-votes)[:top].tolist()

def kg_block(clusters, source):
    feats = augmented if source == "unverified" else verified
    lines = []
    for d in diseases:
        if l2_of[dis_index[d]] in clusters:
            lines.append(f"- {d}: " + ("; ".join(feats.get(d, [])) or "(no verified features)"))
    return "\n".join(lines)

true_l2 = np.array([l2_of[dis_index[d]] for d in test_df["PATHOLOGY"]])
for top in (1, 2):
    hits = [true_l2[p] in route_clusters(test_df.loc[p, "items"], top) for p in range(len(test_df))]
    print(f"Routing accuracy (true disease inside the top-{top} sub-category): {np.mean(hits):.3f}")

## 7. The diagnosis step
The LLM must pick from the allowed diagnoses. We force this with a JSON schema, so the answer is always a valid label (no free-text parsing, no leaked reasoning).

In [ ]:
import difflib

RANK_SCHEMA = {"type": "object",
               "properties": {"ranked_diagnoses": {"type": "array", "minItems": 1, "maxItems": 3,
                                                   "items": {"type": "string", "enum": diseases}}},
               "required": ["ranked_diagnoses"]}
if REASON_FIRST:
    RANK_SCHEMA = {"type": "object",
                   "properties": {"reasoning": {"type": "string"}, **RANK_SCHEMA["properties"]},
                   "required": ["reasoning", "ranked_diagnoses"]}

def case_line(i, idx):
    row = pool_df.loc[idx]
    return f"Case {i}: diagnosis = {row['PATHOLOGY']}. Findings: " + "; ".join(row["items"][:20])

def build_prompt(pos, cfg):
    parts = ["You are a medical diagnosis assistant. Choose the most likely diagnosis for the new patient.",
             "Allowed diagnoses: " + ", ".join(diseases)]
    if cfg["cases"]:
        cases = retrieve_cases(pos, cfg["cases"])
        parts.append("Similar past patients with their confirmed diagnoses:\n" + "\n".join(case_line(i + 1, c) for i, c in enumerate(cases)))
    if cfg["kg"]:
        clusters = route_clusters(test_df.loc[pos, "items"])
        parts.append("Key features of diseases that look similar to this patient (knowledge graph):\n" + kg_block(clusters, cfg["kg"]))
    parts.append("New patient: " + test_df.loc[pos, "text"])
    parts.append("Give the 3 most likely diagnoses, most likely first." if not REASON_FIRST
                 else "First write a brief reasoning, then give the 3 most likely diagnoses, most likely first.")
    return "\n\n".join(parts)

def diagnose(pos, method):
    prompt = build_prompt(pos, METHOD_CONFIG[method])
    data = llm_json(prompt, schema=RANK_SCHEMA, max_tokens=250 if REASON_FIRST else 60)
    raw = data.get("ranked_diagnoses", []) if isinstance(data, dict) else []
    ranked = []
    for name in raw:
        match = name if name in dis_index else (difflib.get_close_matches(str(name), diseases, n=1, cutoff=0.6) or [None])[0]
        if match and match not in ranked:
            ranked.append(match)
    return ranked

# one patient, to see what the prompt and answer look like
p0 = 0
print(build_prompt(p0, METHOD_CONFIG["ours_full"])[:1800], "...\n")
print("True:", test_df.loc[p0, "PATHOLOGY"], "| ours_full predicts:", diagnose(p0, "ours_full"))

## 8. Run all methods (progress bars, resumable)
Every prediction is saved as soon as it is made (`cache_ddx/preds_*.jsonl`). If you stop the notebook or Ollama crashes, run this cell again and it continues where it stopped.
**Time:** roughly 10-40 seconds per patient per method on a CPU laptop, so the default 147 patients x 6 methods can take a few hours. Set `TEST_PER_PATHOLOGY = 1` for a quick trial.

In [ ]:
def pred_file(method):
    return f"{CACHE_DIR}/preds_{LLM_MODEL.replace(':', '_').replace('/', '_')}_{method}.jsonl"

def load_preds(method):
    out = {}
    if os.path.exists(pred_file(method)):
        for line in open(pred_file(method), encoding="utf-8"):
            rec = json.loads(line)
            out[rec["id"]] = rec
    return out

for method in METHODS:
    done = load_preds(method)
    with open(pred_file(method), "a", encoding="utf-8") as f:
        for pos in tqdm(range(len(test_df)), desc=method):
            pid = int(test_df.loc[pos, "orig_idx"])
            if pid in done:
                continue
            t0 = time.time()
            ranked = diagnose(pos, method)
            f.write(json.dumps({"id": pid, "pred": ranked, "seconds": time.time() - t0}) + "\n")
            f.flush()
print("All predictions saved.")

## 9. Metrics
- **Top-1 accuracy (L3)**: the exact diagnosis is first. This is the number comparable with the paper's L3.
- **Top-3 accuracy**, **macro-F1** (treats all pathologies equally), and L2 / L1 accuracy under *our* clusters.
- **95% confidence interval** (Wilson) and a **paired exact test** (McNemar) against two reference methods. Both methods answer the same patients, so the paired test is the fair way to ask "is it really better?".

In [ ]:
from sklearn.metrics import f1_score
from scipy.stats import binomtest

def wilson_ci(correct, n, z=1.96):
    p = correct / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half

all_preds = {m: load_preds(m) for m in METHODS}
ids_all = test_df["orig_idx"].astype(int).tolist()
common = [i for i in ids_all if all(i in all_preds[m] for m in METHODS)]
truth = dict(zip(ids_all, test_df["PATHOLOGY"]))
print(f"Evaluating on {len(common)} patients that every method has answered (of {len(test_df)}).")

def first(pred):
    return pred[0] if pred else None

correct1, rows = {}, []
for m in METHODS:
    y_true = [truth[i] for i in common]
    y_pred = [first(all_preds[m][i]["pred"]) for i in common]
    c1 = np.array([p == t for p, t in zip(y_pred, y_true)])
    c3 = np.array([t in all_preds[m][i]["pred"][:3] for i, t in zip(common, y_true)])
    l2 = np.array([p in dis_index and l2_of[dis_index[p]] == l2_of[dis_index[t]] for p, t in zip(y_pred, y_true)])
    l1 = np.array([p in dis_index and l1_of[dis_index[p]] == l1_of[dis_index[t]] for p, t in zip(y_pred, y_true)])
    lo, hi = wilson_ci(c1.sum(), len(c1))
    correct1[m] = c1
    rows.append({"method": m, "Top-1 (L3)": c1.mean(), "95% CI": f"{lo:.3f}-{hi:.3f}", "Top-3": c3.mean(),
                 "Macro-F1": f1_score(y_true, [p or "none" for p in y_pred], average="macro", zero_division=0),
                 "L2 acc (ours)": l2.mean(), "L1 acc (ours)": l1.mean(),
                 "no valid answer": int(sum(p is None for p in y_pred)),
                 "sec/patient": np.mean([all_preds[m][i]["seconds"] for i in common])})
results_df = pd.DataFrame(rows).set_index("method").round(3)
results_df

In [ ]:
# paired comparison: exact McNemar test (only the patients where two methods disagree matter)
pair_rows = []
for ref in ["naive_rag", "kg_rag_paper_style"]:
    if ref not in METHODS:
        continue
    for m in METHODS:
        if m == ref:
            continue
        only_m   = int((correct1[m] & ~correct1[ref]).sum())
        only_ref = int((~correct1[m] & correct1[ref]).sum())
        p = binomtest(only_m, only_m + only_ref, 0.5).pvalue if only_m + only_ref > 0 else 1.0
        pair_rows.append({"method": m, "vs": ref, "only method right": only_m, "only reference right": only_ref, "p-value": round(p, 3)})
pairs_df = pd.DataFrame(pair_rows)
pairs_df

## 10. Compare with the numbers reported in the paper (context only)

In [ ]:
# DDXPlus L3 accuracy (%) reported in Table 1 of arXiv:2502.04413. Different LLM, prompts and test protocol: NOT a like-for-like comparison.
PAPER_DDXPLUS_L3 = {"Naive RAG + CoT": 56.84, "FS-RAG": 51.40, "FLARE": 31.02, "FL-RAG": 66.78,
                    "DRAGIN": 50.24, "SR-RAG": 52.16, "MedRAG (paper)": 68.01}

print("Reported in the paper (L3 accuracy %):")
for k, v in PAPER_DDXPLUS_L3.items():
    print(f"  {k:18s} {v:6.2f}")
print(f"\nOurs, {len(common)} test patients, LLM = {LLM_MODEL} (Top-1 L3 accuracy %):")
for m in METHODS:
    lo, hi = wilson_ci(correct1[m].sum(), len(correct1[m]))
    print(f"  {m:20s} {100 * correct1[m].mean():6.2f}   (95% CI {100 * lo:.1f}-{100 * hi:.1f})")
print("\nWith few test patients the intervals are wide: a difference of a few points is not evidence of an improvement.")

## 11. Where do the mistakes happen? Save everything

In [ ]:
import matplotlib.pyplot as plt

best = results_df["Top-1 (L3)"].idxmax()
per_path = pd.DataFrame({"pathology": [truth[i] for i in common], "correct": correct1[best]}).groupby("pathology")["correct"].agg(["mean", "count"])
print(f"Hardest pathologies for '{best}':")
print(per_path.sort_values("mean").head(8).round(2).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
acc = results_df["Top-1 (L3)"].values
err = [[a - float(ci.split('-')[0]) for a, ci in zip(acc, results_df["95% CI"])],
       [float(ci.split('-')[1]) - a for a, ci in zip(acc, results_df["95% CI"])]]
ax.bar(results_df.index, acc, yerr=err, capsize=5)
ax.set_ylim(0, 1); ax.set_ylabel("Top-1 accuracy (L3)"); ax.set_title(f"DDXPlus, n={len(common)}, LLM={LLM_MODEL}")
plt.xticks(rotation=25, ha="right"); plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/ddxplus_accuracy.png", dpi=150); plt.show()

results_df.to_csv(f"{RESULTS_DIR}/ddxplus_results.csv")
pairs_df.to_csv(f"{RESULTS_DIR}/ddxplus_paired_tests.csv", index=False)
knn_df.to_csv(f"{RESULTS_DIR}/ddxplus_knn_retrieval.csv", index=False)
json.dump({"settings": {"llm": LLM_MODEL, "embedder": EMBED_MODEL, "test_per_pathology": TEST_PER_PATHOLOGY,
                        "pool_per_pathology": POOL_PER_PATHOLOGY, "top_k_cases": TOP_K_CASES, "kg_top_clusters": KG_TOP_CLUSTERS,
                        "textbook_verification": USE_TEXTBOOK_VERIFICATION, "n_evaluated": len(common),
                        "features_kept_after_verification": f"{n_ver}/{n_aug}"},
           "results": results_df.reset_index().to_dict(orient="records"),
           "paired_tests": pair_rows}, open(f"{RESULTS_DIR}/ddxplus_report.json", "w"), indent=2, default=float)
print("Saved to", RESULTS_DIR)

## 12. What to write in a paper (and what not to)
- Say you **re-implemented a simplified version** of the paper's pipeline and cite it. Describe your two changes (hybrid case retrieval; textbook-verified KG features) and the one-change-at-a-time ablation.
- Report the table with **confidence intervals and the paired tests**, whatever the outcome. If `ours_full` is not clearly better than `kg_rag_paper_style`, say so; a careful negative or inconclusive result is still a valid result.
- Only L3 accuracy is comparable with the paper, and only roughly (different LLM and prompts).
- Run the final numbers with `TEST_PER_PATHOLOGY = 30` and, if you can, repeat with a second seed or LLM.
- **Not implemented here:** the paper's proactive follow-up-question mechanism and its full LLM-driven KG construction (topic aggregation). Check novelty of your changes against the literature before claiming it.